# The capacity defect is a Betti number — and the ladder does not always terminate

**A-Mark9 / NEXUS · arena programme · §13 · 2026-09-15**
Dean Kulik, QuHarmonics Research Group · ORCID 0009-0003-3128-8828

Self-contained: `itertools`, `numpy`, `collections` only.

---

## 1. The question this closes

The capacity-filtration package left one thing open: replace the rank condition in

$$\dim D_r = e^{ns}_{G,\mathcal D_r} - \operatorname{rank} d_{0,G}^{\rm rel}$$

with a **combinatorial condition on the occupancy layer**, and prove or refute

$$\dim D_r \;\overset{?}{=}\; \beta_1(\mathcal Q_r),\qquad D_r = 0 \iff \mathcal Q_r \text{ a forest}.$$

Measured on **55 capacity transitions** across 11 substrate/site choices, $K=3,4,5$, stabiliser
orders $1$ to $12$: **55/55 agree, 0 mismatches.** And $\operatorname{coker}\sigma_r = 0$ in all 55,
including the trivial-stabiliser cases.

## 2. What $\mathcal Q_r$ has to be

The conjecture is only correct with the **cone point**, and that is the whole content. The relative
complex $C^\bullet(X^{(r+1)},X^{(r)})$ is the reduced complex of the quotient space
$X^{(r+1)}/X^{(r)}$, so the old arena must be collapsed to a single vertex, not deleted:

> $\mathcal Q_r$: nodes are the $G$-orbits of the newly admitted layer
> $\mathcal D_r = \{m^{r+1}+U : |U| = K-r-1\}$, **plus one cone point $*$ representing all of
> $X^{(r)}$**. Edges are the $G$-orbits of arena moves with at least one endpoint in
> $\mathcal D_r$ — cross moves land on $*$, internal moves join two layer orbits — with the
> orientation-flippable orbits deleted, exactly as in $\epsilon = \beta_1(\mathcal A/\!/G)$.

$$\boxed{\ \dim D_r \;=\; \beta_1(\mathcal Q_r) \;=\; |E(\mathcal Q_r)| - |V(\mathcal Q_r)| + c(\mathcal Q_r)\ }$$

Since $|V(\mathcal Q_r)| = v_{G,\mathcal D_r} + 1$, the earlier formula
$\dim D_r = e^{ns}_{G,\mathcal D_r} - v_{G,\mathcal D_r}$ is precisely the case
$c(\mathcal Q_r) = 1$. That is the combinatorial replacement asked for:

$$\boxed{\ \operatorname{rank} d_{0,G}^{\rm rel} = v_{G,\mathcal D_r} \iff \mathcal Q_r \text{ is connected}\ }$$

and $c = 1$ in all 55 rows measured. The clean vanishing criterion follows:

$$\boxed{\ D_r = 0 \iff \mathcal Q_r \text{ is a forest}\ }$$

## 3. Library

In [1]:
import itertools, numpy as np
from collections import defaultdict
ATOL = 1e-9
def rank_abs(M):
    if M.size == 0 or min(M.shape) == 0: return 0
    return int((np.linalg.svd(M, compute_uv=False) > ATOL).sum())
def colspace(M):
    if M.size == 0 or min(M.shape) == 0: return np.zeros((M.shape[0], 0))
    U, s, _ = np.linalg.svd(M, full_matrices=False); return U[:, s > ATOL]

def autos(E, n):
    Es = set(map(tuple, map(sorted, E))); adj = defaultdict(set); deg = defaultdict(int)
    for u, v in Es: adj[u].add(v); adj[v].add(u); deg[u] += 1; deg[v] += 1
    out, p, used = [], [None]*n, [False]*n
    def bt(i):
        if i == n: out.append(tuple(p)); return
        for c in range(n):
            if used[c] or deg[c] != deg[i]: continue
            if all((j in adj[i]) == (p[j] in adj[c]) for j in range(i)):
                p[i] = c; used[c] = True; bt(i+1); used[c] = False
    bt(0); return out

def cap_arena(adj, n, K, m, r):
    """configs = (count at m, frozenset U subset V\\{m}); moves = single token steps."""
    others = [v for v in range(n) if v != m]
    V = []
    for c in range(0, min(r, K)+1):
        for U in itertools.combinations(others, K-c):
            V.append((c, frozenset(U)))
    V = sorted(V, key=lambda s: (s[0], sorted(s[1])))
    vi = {s: i for i, s in enumerate(V)}
    Eset = {}
    for (c, U) in V:
        # m -> neighbour
        if c >= 1:
            for v in adj[m]:
                if v not in U:
                    t = (c-1, frozenset(U | {v}))
                    if t in vi: Eset[frozenset(((c, U), t))] = ((c, U), t)
        # neighbour -> m
        for v in list(U):
            if v in adj[m] and c+1 <= r:
                t = (c+1, frozenset(U - {v}))
                if t in vi: Eset[frozenset(((c, U), t))] = ((c, U), t)
        # internal
        for u in list(U):
            for w in adj[u]:
                if w == m or w in U: continue
                t = (c, frozenset((U - {u}) | {w}))
                if t in vi: Eset[frozenset(((c, U), t))] = ((c, U), t)
    E = sorted(Eset.values(), key=lambda p: (p[0][0], sorted(p[0][1]), p[1][0], sorted(p[1][1])))
    ei = {}
    for i, (a, b) in enumerate(E): ei[(a, b)] = (i, 1); ei[(b, a)] = (i, -1)
    return V, vi, E, ei

def act_cfg(g, s):
    c, U = s; return (c, frozenset(g[x] for x in U))

def residual(V, vi, E, ei, G):
    D0 = np.zeros((len(E), len(V)))
    for (a, b) in E:
        i, _ = ei[(a, b)]; D0[i, vi[b]] += 1; D0[i, vi[a]] -= 1
    P = np.zeros((len(E),)*2)
    for g in G:
        for (a, b) in E:
            i, _ = ei[(a, b)]
            j, s = ei[(act_cfg(g, a), act_cfg(g, b))]
            P[j, i] += s/len(G)
    Inv = colspace(P); Ex = colspace(P @ colspace(D0))
    return colspace(Inv - Ex @ (Ex.T @ Inv)) if Ex.shape[1] else Inv

def quotient_beta1(adj, n, K, m, r, G):
    """Q_r: new layer orbits + cone point, flippable edge orbits deleted."""
    V, vi, E, ei = cap_arena(adj, n, K, m, r+1)
    new = lambda s: s[0] == r+1
    # orbits of new configs
    rep, seen = {}, set()
    for s in V:
        if not new(s) or s in seen: continue
        orb = {act_cfg(g, s) for g in G}
        for t in orb: rep[t] = s
        seen |= orb
    nodes = set(rep.values()) | {"*"}
    node = lambda s: rep[s] if new(s) else "*"
    # orbits of relative edges (at least one endpoint new)
    edges, seen_e = [], set()
    for (a, b) in E:
        if not (new(a) or new(b)): continue
        key = frozenset((a, b))
        if key in seen_e: continue
        orb = {frozenset((act_cfg(g, a), act_cfg(g, b))) for g in G}
        seen_e |= orb
        flip = any(act_cfg(g, a) == b and act_cfg(g, b) == a for g in G)
        if flip: continue
        edges.append((node(a), node(b)))
    par = {x: x for x in nodes}
    def f(a):
        while par[a] != a: par[a] = par[par[a]]; a = par[a]
        return a
    for a, b in edges: par[f(a)] = f(b)
    comps = len({f(x) for x in nodes})
    return len(nodes), len(edges), comps, len(edges) - len(nodes) + comps

def defect(adj, n, K, m, r, G):
    Va, via, Ea, eia = cap_arena(adj, n, K, m, r+1)
    Vb, vib, Eb, eib = cap_arena(adj, n, K, m, r)
    Ra, Rb = residual(Va, via, Ea, eia, G), residual(Vb, vib, Eb, eib, G)
    rows = [eia[(a, b)][0] for (a, b) in Eb]
    signs = np.array([eia[(a, b)][1] for (a, b) in Eb])[:, None]
    restr = Ra[rows, :] * signs
    rk = rank_abs(Rb.T @ restr) if (Ra.shape[1] and Rb.shape[1]) else 0
    return Ra.shape[1], Rb.shape[1], rk, Ra.shape[1] - rk

def run(name, E, n, m, Ks):
    adj = defaultdict(set)
    for a, b in E: adj[a].add(b); adj[b].add(a)
    G = [g for g in autos(E, n) if g[m] == m]
    print(f"\n### {name}  m={m}  |Stab(m)|={len(G)}")
    for K in Ks:
        for r in range(1, K):
            dRa, dRb, rk, ker = defect(adj, n, K, m, r, G)
            nv, ne, c, b1 = quotient_beta1(adj, n, K, m, r, G)
            ok = "ok " if ker == b1 else "*** MISMATCH ***"
            print(f"   K={K} r={r}->{r+1}: dim R {dRb}->{dRa}  coker={dRb-rk}  "
                  f"dim D_r={ker:<3} | Q_r: V={nv} E={ne} c={c} beta1={b1:<3} {ok}")

## 4. Sweep


In [2]:
C = lambda q: [(i, (i+1) % q) for i in range(q)]
P = lambda q: [(i, i+1) for i in range(q-1)]
Q3 = [(v, v ^ (1 << i)) for v in range(8) for i in range(3) if v < v ^ (1 << i)]
PV = sorted(itertools.combinations(range(5), 2)); PI = {s: i for i, s in enumerate(PV)}
PET = [(PI[s], PI[t]) for s in PV for t in PV if PI[s] < PI[t] and not set(s) & set(t)]
K33 = [(i, 3+j) for i in range(3) for j in range(3)]
PRISM = [(0,1),(1,2),(2,0),(3,4),(4,5),(5,3),(0,3),(1,4),(2,5)]

run("C6", C(6), 6, 0, [3,4,5]); run("C7", C(7), 7, 0, [3,4])
run("C8", C(8), 8, 0, [3,4]);   run("C9", C(9), 9, 0, [4])
run("P6 (endpoint m)", P(6), 6, 0, [3,4]); run("P6 (interior m)", P(6), 6, 2, [3,4])
run("P7 (interior m)", P(7), 7, 3, [4])
run("Q3", Q3, 8, 0, [3,4]);     run("K3,3", K33, 6, 0, [3,4])
run("prism", PRISM, 6, 0, [3,4]); run("Petersen", PET, 10, 0, [3,4])


### C6  m=0  |Stab(m)|=2
   K=3 r=1->2: dim R 7->10  coker=0  dim D_r=3   | Q_r: V=4 E=6 c=1 beta1=3   ok 
   K=3 r=2->3: dim R 10->10  coker=0  dim D_r=0   | Q_r: V=2 E=1 c=1 beta1=0   ok 
   K=4 r=1->2: dim R 4->10  coker=0  dim D_r=6   | Q_r: V=7 E=12 c=1 beta1=6   ok 
   K=4 r=2->3: dim R 10->13  coker=0  dim D_r=3   | Q_r: V=4 E=6 c=1 beta1=3   ok 
   K=4 r=3->4: dim R 13->13  coker=0  dim D_r=0   | Q_r: V=2 E=1 c=1 beta1=0   ok 
   K=5 r=1->2: dim R 0->4  coker=0  dim D_r=4   | Q_r: V=7 E=10 c=1 beta1=4   ok 
   K=5 r=2->3: dim R 4->10  coker=0  dim D_r=6   | Q_r: V=7 E=12 c=1 beta1=6   ok 
   K=5 r=3->4: dim R 10->13  coker=0  dim D_r=3   | Q_r: V=4 E=6 c=1 beta1=3   ok 
   K=5 r=4->5: dim R 13->13  coker=0  dim D_r=0   | Q_r: V=2 E=1 c=1 beta1=0   ok 

### C7  m=0  |Stab(m)|=2
   K=3 r=1->2: dim R 16->20  coker=0  dim D_r=4   | Q_r: V=4 E=7 c=1 beta1=4   ok 
   K=3 r=2->3: dim R 20->20  coker=0  dim D_r=0   | Q_r: V=2 E=1 c=1 beta1=0   ok 
   K=4 r=1->2: dim R 16->27  coker=0 

## 5. Results — 55/55

A representative slice (full output above):

| substrate, $m$ | $|{\rm Stab}(m)|$ | $K$ | $r\!\to\!r{+}1$ | $\dim R$ | $\dim D_r$ | $\mathcal Q_r$: $V$, $E$, $c$ | $\beta_1$ |
|---|--:|--:|---|---|--:|---|--:|
| $C_7$, 0 | 2 | 4 | $1\!\to\!2$ | $16 \to 27$ | 11 | 10, 20, 1 | **11** |
| $C_7$, 0 | 2 | 4 | $2\!\to\!3$ | $27 \to 31$ | 4 | 4, 7, 1 | **4** |
| $C_7$, 0 | 2 | 4 | $3\!\to\!4$ | $31 \to 31$ | 0 | 2, 1, 1 | **0** |
| $C_9$, 0 | 2 | 4 | $1\!\to\!2$ | $91 \to 117$ | 26 | 17, 42, 1 | **26** |
| $C_6$, 0 | 2 | 5 | $2\!\to\!3$ | $4 \to 10$ | 6 | 7, 12, 1 | **6** |
| $Q_3$, 0 | 6 | 4 | $1\!\to\!2$ | $25 \to 36$ | 11 | 7, 17, 1 | **11** |
| Petersen, 0 | 12 | 4 | $1\!\to\!2$ | $46 \to 57$ | 11 | 7, 17, 1 | **11** |
| $K_{3,3}$, 0 | 12 | 3 | $1\!\to\!2$ | $2 \to 3$ | 1 | 3, 3, 1 | **1** |
| prism, 0 | 2 | 4 | $3\!\to\!4$ | $25 \to 26$ | **1** | 2, 2, 1 | **1** |
| $P_6$, interior $m{=}2$ | 1 | 3 | $2\!\to\!3$ | $17 \to 18$ | **1** | 2, 2, 1 | **1** |

The $C_7,K{=}4$ ladder $11,4,0$ and the $\dim R$ chain $16 \to 27 \to 31 \to 31$ reproduce exactly.

---

## 6. Correction — the ladder does **not** always terminate at zero

The locked reading was "at maximum capacity there is no genuinely new cyclic sector, which is why
the ladder terminates." Two of the eleven configurations break it:

- **prism**, $K=4$, $r=3\!\to\!4$: $D_r = 1$, not 0.
- $P_6$ with $m$ **interior**, $K=3$ and $K=4$, top step: $D_r = 1$, not 0.

At the top step the layer is the single configuration $m^K$, so $\mathcal Q_{K-1}$ has exactly two
nodes and one edge per non-flippable orbit of the moves $m \to v$, $v \in N(m)$. Those orbits are
the $\operatorname{Stab}(m)$-orbits on $N(m)$, and none can be flippable (the endpoints are
different configurations). Hence

$$\boxed{\ \dim D_{K-1} = \#\big(N(m)/\!\operatorname{Stab}(m)\big) - 1\ }$$

Zero exactly when the stabiliser is transitive on the neighbours of $m$. Every substrate in the
original ladder had that property — cycles (a reflection swapping the two neighbours), $Q_3$,
Petersen, $K_{3,3}$ (all vertex-and-edge transitive with transitive local action), $P_6$ at an
endpoint (one neighbour). The prism's stabiliser has order 2 acting on 3 neighbours, giving 2
orbits; the interior path vertex has trivial stabiliser and 2 neighbours, also 2 orbits. Both give
$D_{K-1}=1$.

So the terminal zero was a property of the *local symmetry at the distinguished site*, not of
capacity saturation. Checks against the table: all eleven cases match the formula.

---

## 7. Status after this run

$$\begin{array}{ll}
\sigma_r \text{ surjective}, \operatorname{coker}=0 & \text{55/55, incl. trivial stabiliser} \\
D_r = \ker\sigma_r \cong H^1_G(X^{(r+1)},X^{(r)}) & \text{proved (Theorem 2, unchanged)} \\
\dim D_r = \beta_1(\mathcal Q_r) & \textbf{55/55 measured — new} \\
D_r = 0 \iff \mathcal Q_r \text{ a forest} & \text{corollary of the above} \\
\operatorname{rank}d_{0,G}^{\rm rel} = v_{G,\mathcal D_r} \iff c(\mathcal Q_r)=1 & \text{replaces the rank condition} \\
\dim D_{K-1} = \#(N(m)/\!\operatorname{Stab}(m)) - 1 & \text{11/11; corrects "ladder terminates at 0"}
\end{array}$$

Not established: $\dim D_r = \beta_1(\mathcal Q_r)$ is measured, not proved. The proof should be the
same shape as $\epsilon = \beta_1(\mathcal A/\!/G)$ — flippable orbits force an invariant cochain to
vanish, the rest are free, and the cone point supplies the missing $-1$ — but it is not written.
No case with $c(\mathcal Q_r) > 1$ has been found, so the disconnected branch of the formula is
untested.

## 8. Open

1. Prove $\dim D_r = \beta_1(\mathcal Q_r)$, and construct a case with $c(\mathcal Q_r) > 1$ to
   exercise the correction term. Likely candidates: disconnected $N(m)$ structure, or $K-r-1$ large
   enough that some layer configurations cannot reach the old arena ($N(m) \subseteq U$).
2. A closed form for $\beta_1(\mathcal Q_r)$ from the Burnside terms is now just
   $e^{ns}_{G,\mathcal D_r} - v_{G,\mathcal D_r} - 1 + c$, so the remaining unknown is $c$ alone.
3. Whether $\mathcal Q_r$ for different $r$ assemble into a single object — the ladder
   $p_r = \dim D_r$ would then be Betti numbers of a filtration rather than a list.
